# ex06 · 综合实战：迷你训练流水线（贯穿 2.1-2.5）

> **题型：代码补全**。主框架已经搭好，每个 `# TODO` 留空，对应本章某节的知识点。
> 全部补全后，整条流水线能跑通——它就是第 3 章「训练循环」的雏形。
>
> **知识地图**（每个 TODO 对应哪节）：
>
> | 关卡 | 任务 | 对应章节 |
> |---|---|---|
> | 1 | 创建数据张量 | 2.1 |
> | 2 | 特征标准化 | 2.1 + 2.3（广播、降维） |
> | 3 | 线性前向计算 | 2.3（矩阵乘法 + 广播） |
> | 4 | 损失 + 反向传播 | 2.4 + 2.5（自动微分） |
> | 5 | 手动梯度下降一步 | 2.5 延伸（第 3 章伏笔） |
>
> 答案在 `solutions/ex06-答案.md`。

**背景故事**：你有 4 个样本、每个样本 3 个特征（比如 4 套房子的 3 个指标），想用最简单的线性模型 `预测值 = 特征点乘权重 + 偏置` 去拟合标签 y（房价）。训练 = 让预测值和标签的差距（损失）变小。

In [29]:
import torch

## 关卡 1 · 创建数据（2.1）

特征矩阵 `X` 形状 `(4, 3)`：4 个样本 × 3 个特征，用 `arange` 填充 0~11；标签 `y` 形状 `(4,)`，随机数。

In [3]:
help(torch.arange)

Help on built-in function arange in module torch:

arange(...)
    arange(start=0, end, step=1, *, out=None, dtype=None, layout=torch.strided, device=None, requires_grad=False) -> Tensor
    
    Returns a 1-D tensor of size :math:`\left\lceil \frac{\text{end} - \text{start}}{\text{step}} \right\rceil`
    with values from the interval ``[start, end)`` taken with common difference
    :attr:`step` beginning from `start`.
    
    Note: When using floating-point dtypes (especially reduced precision types like ``bfloat16``),
    the results may be affected by floating-point rounding behavior. Some values in the sequence
    might not be exactly representable in certain floating-point formats, which can lead to
    repeated values or unexpected rounding. For precise sequences, it is recommended to use
    integer dtypes instead of floating-point dtypes.
    
    Note that non-integer :attr:`step` is subject to floating point rounding errors when
    comparing against :attr:`end`; to avoid

In [46]:
# TODO 1.1: 用 torch.arange 创建 0~11 共 12 个数，reshape 成 (4, 3)，dtype 用 float32
# X = 
X = torch.arange(12.).reshape(4, 3)
# TODO 1.2: 用 torch.randn 创建标签 y，形状 (4,)
# y = 
y = torch.randn(4,)
# ---- 验证（补完上面的 TODO 再运行）----
try:
    print('X.shape =', X.shape)   # 期望 torch.Size([4, 3])
    print('X.dtype =', X.dtype)   # 期望 torch.float32
    print('y.shape =', y.shape)   # 期望 torch.Size([4])
except NameError:
    print('⚠ 先完成 TODO 1.1 / 1.2')

X.shape = torch.Size([4, 3])
X.dtype = torch.float32
y.shape = torch.Size([4])


ps: pytorch中浮点默认 float32；整数默认 int64

## 关卡 2 · 特征标准化（2.1 + 2.3）

对 `X` 的**每一列**做 z-score 标准化：`(X - 列均值) / 列标准差`。
目的是让 3 个特征的数值在同一个量级，避免「数值大的特征主导梯度」。

提示：`mean(dim=0, keepdim=True)` 得到形状 `(1, 3)` 的列均值；广播会自动把它「复制」到 4 行。

In [47]:
# TODO 2.1: 每列的均值，形状 (1, 3)
# col_mean = 
col_mean = X.mean(dim = 0, keepdim = True)
# TODO 2.2: 每列的标准差，形状 (1, 3)
# col_std = 
col_std = X.std(dim = 0, keepdim = True)
# TODO 2.3: 标准化
# X_norm = 
X_norm = (X - col_mean) / col_std
# ---- 验证 ----
try:
    print('X_norm.shape =', X_norm.shape)          # 期望 (4, 3)
    print('每列均值 ≈', X_norm.mean(dim=0))         # 期望全 ≈ 0
    print('每列标准差 ≈', X_norm.std(dim=0))        # 期望全 ≈ 1
except NameError:
    print('⚠ 先完成 TODO 2.1 ~ 2.3')

X_norm.shape = torch.Size([4, 3])
每列均值 ≈ tensor([0., 0., 0.])
每列标准差 ≈ tensor([1.0000, 1.0000, 1.0000])


## 关卡 3 · 线性前向计算（2.3）

最简单的模型：`y_pred = X_norm @ w + b`。
其中 `w` 是权重向量（形状 `(3,)`），`b` 是偏置（形状 `(1,)`）。

注意：`w`、`b` 是我们想**学习**的参数，创建时要开梯度追踪。

In [8]:
help(torch.tensor)

Help on built-in function tensor in module torch:

tensor(...)
    tensor(data, *, dtype=None, device=None, requires_grad=False, pin_memory=False) -> Tensor
    
    Constructs a tensor with no autograd history (also known as a "leaf tensor", see :doc:`/notes/autograd`) by copying :attr:`data`.
    
    .. warning::
    
        When working with tensors prefer using :func:`torch.Tensor.clone`,
        :func:`torch.Tensor.detach`, and :func:`torch.Tensor.requires_grad_` for
        readability. Letting `t` be a tensor, ``torch.tensor(t)`` is equivalent to
        ``t.detach().clone()``, and ``torch.tensor(t, requires_grad=True)``
        is equivalent to ``t.detach().clone().requires_grad_(True)``.
    
    .. seealso::
    
        :func:`torch.as_tensor` preserves autograd history and avoids copies where possible.
        :func:`torch.from_numpy` creates a tensor that shares storage with a NumPy array.
    
    Args:
        data (array_like): Initial data for the tensor. Can be a li

In [48]:
# TODO 3.1: 创建权重 w（形状 (3,)，requires_grad=True）和偏置 b（形状 (1,)，requires_grad=True）
# w = 
# b = 
w = torch.randn(3, requires_grad = True)
b = torch.randn(1, requires_grad = True)
# TODO 3.2: 前向计算 y_pred = X_norm @ w + b
# y_pred = 
y_pred = X @ w + b
# ---- 验证 ----
try:
    print('y_pred.shape =', y_pred.shape)      # 期望 torch.Size([4])
    print('w.requires_grad =', w.requires_grad)   # 期望 True
except NameError:
    print('⚠ 先完成 TODO 3.1 / 3.2')

y_pred.shape = torch.Size([4])
w.requires_grad = True


In [38]:
y_pred = X_norm @ w + b

## 关卡 4 · 损失与反向传播（2.4 + 2.5）

均方误差（MSE）损失：预测值和标签差值的平方，再取平均。
然后调用 `backward()`，梯度会自动算好，存在 `w.grad` 和 `b.grad` 里。

**思考**：`loss.backward()` 之后，`w.grad` 的含义是什么？（提示：损失对 w 的偏导，即「w 动一点点，loss 会动多少」）

In [14]:
help(torch.mean)

Help on built-in function mean in module torch:

mean(...)
    mean(input, *, dtype=None) -> Tensor
    
    .. note::
        If the `input` tensor is empty, ``torch.mean()`` returns ``nan``.
        This behavior is consistent with NumPy and follows the definition
        that the mean over an empty set is undefined.
    
    
    Returns the mean value of all elements in the :attr:`input` tensor. Input must be floating point or complex.
    
    Args:
        input (Tensor):
          the input tensor, either of floating point or complex dtype
    
    Keyword args:
        dtype (:class:`torch.dtype`, optional): the desired data type of returned tensor.
            If specified, the input tensor is casted to :attr:`dtype` before the operation
            is performed. This is useful for preventing data type overflows. Default: None.
    
    Example::
    
        >>> a = torch.randn(1, 3)
        >>> a
        tensor([[ 0.2294, -0.5481,  1.3288]])
        >>> torch.mean(a)
       

In [49]:
# TODO 4.1: 计算均方误差 loss = ((y_pred - y) 的平方) 的平均值
# loss = 
loss = ((y_pred - y) **2).mean()
# TODO 4.2: 反向传播
# loss.backward()
loss.backward()
# ---- 验证 ----
try:
    print('loss =', loss.item())             # 一个正数
    print('w.grad.shape =', w.grad.shape)    # 期望 torch.Size([3])
    print('b.grad =', b.grad)                # 一个标量张量
except NameError:
    print('⚠ 先完成 TODO 4.1 / 4.2')

loss = 8.691071510314941
w.grad.shape = torch.Size([3])
b.grad = tensor([1.8300])


## 关卡 5 · 手动梯度下降一步（2.5 延伸）

训练的本质：**沿负梯度方向更新参数**，一步走多远由学习率 `lr` 决定。

$$w \leftarrow w - lr \cdot \frac{\partial loss}{\partial w}$$

（第 3 章会用 `optimizer.step()` 自动完成这一步，这里手动走一遍感受它。）

In [50]:
lr = 0.01

# TODO 5.1: 更新 w（沿负梯度方向走一小步）
# w = 
#w = w - lr * w.grad
# TODO 5.2: 同样更新 b
# b = 
#b = b - lr * b.grad
# ---- 修改，为了能重复运行 ----
with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()
# ---- 验证 ----
try:
    print('更新后的 w =', w)      # 和初始随机值略有不同
    print('更新后的 b =', b)
except NameError:
    print('⚠ 先完成 TODO 5.1 / 5.2')

更新后的 w = tensor([ 1.3743, -1.1422, -0.3333], requires_grad=True)
更新后的 b = tensor([-0.7343], requires_grad=True)


ps. 实际上这种跑法只有第一次能跑通，参照前面的叶子节点的知识，w已经被覆盖，不再具有叶子节点的性质，所以无法循环跑这个代码

In [45]:
# --- 合体写一个训练流程 ---
X = torch.randn(4, 3)
w = torch.randn(3, requires_grad = True)
b = torch.randn(1, requires_grad = True)
y = torch.randn(4,) #标签量
Epoch = 30
lr = 1
for i in range(Epoch):
    y_pred = X @ w + b
    loss = ((y_pred - y) ** 2).mean()
    print(f"第 {i} 次 w = {w}")
    print(f"第 {i} 次 b = {b}")
    print(f"第 {i} 次 loss = {loss}")
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

第 0 次 w = tensor([-0.4437, -1.0854, -0.1025], requires_grad=True)
第 0 次 b = tensor([1.2330], requires_grad=True)
第 0 次 loss = 3.349245548248291
第 1 次 w = tensor([ 0.7753,  2.1930, -2.2194], requires_grad=True)
第 1 次 b = tensor([-0.7039], requires_grad=True)
第 1 次 loss = 16.74423599243164
第 2 次 w = tensor([-0.0556, -5.6199,  4.0827], requires_grad=True)
第 2 次 b = tensor([3.0595], requires_grad=True)
第 2 次 loss = 101.78670501708984
第 3 次 w = tensor([  1.4099,  14.6816, -11.4606], requires_grad=True)
第 3 次 b = tensor([-4.3566], requires_grad=True)
第 3 次 loss = 633.991455078125
第 4 次 w = tensor([ -0.8750, -37.1257,  27.1621], requires_grad=True)
第 4 次 b = tensor([11.8055], requires_grad=True)
第 4 次 loss = 3976.356201171875
第 5 次 w = tensor([  3.1202,  94.0235, -69.2045], requires_grad=True)
第 5 次 b = tensor([-25.3993], requires_grad=True)
第 5 次 loss = 24989.34375
第 6 次 w = tensor([  -4.5408, -236.5197,  171.8113], requires_grad=True)
第 6 次 b = tensor([63.4545], requires_grad=True)
第 6 次 lo

## 附加思考题（不写代码）

1. 如果跳过关卡 2（不标准化，直接用 `X` 前向），梯度会怎样变化？（提示：`X` 第三列数值最大）
   `X` 第三列数值最大 → 它对应的权重 `w₃` 梯度最大、更新最快，而 `w₀` 几乎不动。模型被大数值特征"带偏"，训练慢且不稳定。标准化后所有特征梯度同量级，训练平衡。
    
3. 如果把 `lr` 改成 1.0 甚至 10，参数更新会怎样？
    下降过快，导致loss出现先激增再迅速下降的过程。学习率不能太大，避免loss剧烈震荡
   
4. 把关卡 3~5 从头到尾再跑一遍，loss 会变小吗？（这就是训练循环：重复前向→反向→更新）
    标准情况下是会的
   
**全部跑通 + 答出 3 个思考题 = 第 2 章毕业。**